In [1]:
%pip install -q \
    "lerobot[smolvla,dataset]==0.6.1" \
    "transformers==5.5.0" \
    "accelerate>=1.12.0" \
    "safetensors>=0.5" \
    "pillow>=10" \
    "pandas>=2" \
    "av"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nilearn 0.14.1 requires packaging>=26.0.0, but you have packaging 25.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, gc, json, time, random, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers

from huggingface_hub import hf_hub_download
from lerobot.datasets.lerobot_dataset import LeRobotDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert torch.cuda.is_available(), "Enable Kaggle GPU"

DEVICE = torch.device("cuda:0")
DATASET_ID = "nauq-hna/clean_290_60annot_v3"
SMOL_ID = "lerobot/smolvla_base"

STEPS = 10_000
LR_A = 1e-4
CHUNK = 50
DIM = 6
FPS = 30
FLOW_STEPS = 10

VAL_EVERY = 1000
LOG_EVERY = 100
WINDOWS_PER_EP = 5
NOISE_SEEDS = [0, 1, 2]

CAM_KEYS = [
    "observation.images.cam_head",
    "observation.images.cam_side",
    "observation.images.cam_wrist",
]

INSTRUCTION = (
    "pick up the orange cube and place it into the bowl"
)

OUT = Path("/kaggle/working/VLA_A_seed42")
OUT.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# FIXED RANDOM SPLIT — NEVER MODIFY AFTER TRAINING
# --------------------------------------------------

meta_path = hf_hub_download(
    DATASET_ID,
    "meta/info.json",
    repo_type="dataset",
    token=HF_TOKEN,
)

with open(meta_path) as f:
    metadata = json.load(f)

TOTAL_EPS = int(metadata["total_episodes"])
FPS = float(metadata["fps"])

assert TOTAL_EPS == 290, TOTAL_EPS

rng = np.random.default_rng(SEED)
permutation = rng.permutation(TOTAL_EPS).tolist()

TRAIN_EPS = sorted(permutation[:100])
VAL_EPS = sorted(permutation[100:130])
TEST_EPS = sorted(permutation[130:190])
UNUSED_EPS = sorted(permutation[190:])

assert len(TRAIN_EPS) == 100
assert len(VAL_EPS) == 30
assert len(TEST_EPS) == 60
assert not (
    set(TRAIN_EPS) & set(VAL_EPS)
    or set(TRAIN_EPS) & set(TEST_EPS)
    or set(VAL_EPS) & set(TEST_EPS)
)

split_df = pd.DataFrame(
    [
        {"episode": ep, "split": split}
        for split, eps in [
            ("train", TRAIN_EPS),
            ("val", VAL_EPS),
            ("test", TEST_EPS),
            ("unused", UNUSED_EPS),
        ]
        for ep in eps
    ]
).sort_values("episode").reset_index(drop=True)

split_df.to_csv(
    OUT / "split_seed42.csv",
    index=False,
)

SPLIT_HASH = hashlib.sha256(
    (OUT / "split_seed42.csv").read_bytes()
).hexdigest()

print("Train:", len(TRAIN_EPS))
print("Val:", len(VAL_EPS))
print("Test:", len(TEST_EPS))
print("Unused:", len(UNUSED_EPS))
print("Split SHA256:", SPLIT_HASH)

# --------------------------------------------------
# DATASET AND NORMALIZATION — TRAIN ONLY
# --------------------------------------------------

deltas = {
    "action": [
        round(i / FPS, 6)
        for i in range(CHUNK)
    ]
}

ds = LeRobotDataset(
    DATASET_ID,
    episodes=TRAIN_EPS + VAL_EPS,
    delta_timestamps=deltas,
    revision="main",
    token=HF_TOKEN,
    download_videos=True,
)

rows = ds.select_columns([
    "episode_index",
    "frame_index",
    "action",
    "observation.state",
]).to_pandas().reset_index(drop=True)

train_rows = rows[
    rows.episode_index.isin(TRAIN_EPS)
]

a_arr = np.stack(
    train_rows["action"].to_numpy()
).astype(np.float32)

s_arr = np.stack(
    train_rows["observation.state"].to_numpy()
).astype(np.float32)

assert a_arr.shape[1] == DIM
assert s_arr.shape[1] == DIM

a_mean = a_arr.mean(axis=0)
a_std = np.maximum(a_arr.std(axis=0), 1e-4)
s_mean = s_arr.mean(axis=0)
s_std = np.maximum(s_arr.std(axis=0), 1e-4)

np.savez(
    OUT / "normalization_seed42.npz",
    AM=a_mean, AS=a_std,
    SM=s_mean, SS=s_std,
)

AM = torch.tensor(a_mean, device=DEVICE)
AS = torch.tensor(a_std, device=DEVICE)
SM = torch.tensor(s_mean, device=DEVICE)
SS = torch.tensor(s_std, device=DEVICE)

valid = {}

for ep in TRAIN_EPS + VAL_EPS:
    group = rows[rows.episode_index == ep]
    last = int(group.frame_index.max())

    candidates = group[
        group.frame_index <= last - CHUNK + 1
    ]

    valid[ep] = candidates.index.to_numpy(
        dtype=int
    )

    assert len(valid[ep]) > 0, ep

# Episode-balanced, deterministic training samples.
sample_rng = np.random.default_rng(SEED)

TRAIN_PLAN = []

for step in range(1, STEPS + 1):
    ep = TRAIN_EPS[(step - 1) % len(TRAIN_EPS)]
    idx = int(sample_rng.choice(valid[ep]))

    TRAIN_PLAN.append({
        "step": step,
        "episode": ep,
        "index": idx,
    })

pd.DataFrame(TRAIN_PLAN).to_csv(
    OUT / "training_plan.csv",
    index=False,
)

# One fixed valid window per validation episode.
VAL_INDICES = [
    int(valid[ep][len(valid[ep]) // 2])
    for ep in VAL_EPS
]

def chw_image(x):
    x = torch.as_tensor(x).float()

    if x.ndim != 3:
        raise ValueError(tuple(x.shape))

    if x.shape[-1] in (3, 4) and x.shape[0] != 3:
        x = x.permute(2, 0, 1)

    x = x[:3].contiguous()

    if x.max().item() > 1.5:
        x = x / 255.0

    return x.clamp(0, 1)

def get_sample(dataset, idx):
    raw = dataset[int(idx)]

    state = torch.as_tensor(
        raw["observation.state"],
        dtype=torch.float32,
        device=DEVICE,
    ).reshape(1, DIM)

    action = torch.as_tensor(
        raw["action"],
        dtype=torch.float32,
        device=DEVICE,
    ).reshape(1, CHUNK, DIM)

    images = {
        k: chw_image(raw[k]).unsqueeze(0).to(DEVICE)
        for k in CAM_KEYS
    }

    instruction = raw.get("task", INSTRUCTION)

    if not isinstance(instruction, str) or not instruction.strip():
        instruction = INSTRUCTION

    return {
        "state": (state - SM) / SS,
        "target": (action - AM) / AS,
        "raw_action": action,
        "images": images,
        "task": instruction,
    }

def fixed_noise_time(seed, action_dim=DIM):
    g = torch.Generator(
        device=DEVICE
    ).manual_seed(int(seed))

    noise = torch.randn(
        (1, CHUNK, action_dim),
        generator=g,
        device=DEVICE,
    )

    tau_value = (
        np.random.default_rng(
            20000 + int(seed)
        ).beta(1.5, 1.0) * 0.999 + 0.001
    )

    tau = torch.tensor(
        [tau_value],
        device=DEVICE,
        dtype=torch.float32,
    )

    return noise, tau

def get_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    return total, trainable

print("[PASS] DATA, SPLIT AND NORMALIZATION READY")

PyTorch: 2.11.0+cu128
Transformers: 5.5.0
GPU: Tesla T4


info.json:   0%|          | 0.00/5.04k [00:00<?, ?B/s]

Train: 100
Val: 30
Test: 60
Unused: 100
Split SHA256: 70319105efb17fa49e7774f8649239306a753443a62871d2c76a130288f5f9a0


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

[PASS] DATA, SPLIT AND NORMALIZATION READY


In [3]:
from lerobot.configs import PolicyFeature, FeatureType
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.utils.constants import (
    OBS_LANGUAGE_TOKENS,
    OBS_LANGUAGE_ATTENTION_MASK,
)

sample = get_sample(ds, int(valid[TRAIN_EPS[0]][0]))

policy = SmolVLAPolicy.from_pretrained(
    SMOL_ID
).to(DEVICE)

image_shape = tuple(
    sample["images"][CAM_KEYS[0]].shape[1:]
)

policy.config.input_features = {
    "observation.state": PolicyFeature(
        type=FeatureType.STATE,
        shape=(DIM,),
    ),
    **{
        k: PolicyFeature(
            type=FeatureType.VISUAL,
            shape=image_shape,
        )
        for k in CAM_KEYS
    },
}

policy.config.output_features = {
    "action": PolicyFeature(
        type=FeatureType.ACTION,
        shape=(DIM,),
    ),
}

policy.config.chunk_size = CHUNK
policy.config.n_action_steps = CHUNK
policy.config.num_steps = FLOW_STEPS
policy.config.empty_cameras = 0
policy.config.adapt_to_pi_aloha = False
policy.config.rtc_config = None

tokenizer = (
    policy.model.vlm_with_expert.processor.tokenizer
)

def a_batch(s, include_action=True):
    tokens = tokenizer(
        s["task"].rstrip() + "\n",
        return_tensors="pt",
        padding="max_length",
        max_length=policy.config.tokenizer_max_length,
        truncation=True,
    )

    b = {
        "observation.state": s["state"],
        **s["images"],
        OBS_LANGUAGE_TOKENS: tokens["input_ids"].to(DEVICE),
        OBS_LANGUAGE_ATTENTION_MASK: (
            tokens["attention_mask"].to(DEVICE).bool()
        ),
    }

    if include_action:
        b["action"] = s["target"]

    return b

# FP32 trainable weights, frozen weights unchanged.
for p in policy.parameters():
    if p.requires_grad and p.is_floating_point():
        if p.dtype != torch.float32:
            p.data = p.data.float()

trainable = [
    p for p in policy.parameters()
    if p.requires_grad
]

assert trainable
assert all(p.dtype == torch.float32 for p in trainable)

optim_a = torch.optim.AdamW(
    trainable,
    lr=LR_A,
)

A_TOTAL, A_TRAINABLE = get_params(policy)

print("Total parameters:", A_TOTAL)
print("Trainable parameters:", A_TRAINABLE)

with torch.no_grad(), torch.autocast(
    "cuda", dtype=torch.float16
):
    loss, _ = policy(a_batch(sample))

assert torch.isfinite(loss)

print("[PASS] SmolVLA forward loss:", loss.item())

config.json:   0%|          | 0.00/2.30k [00:00<?, ?B/s]

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.03GB            

...

tokenizer.json:   0%|          | 0.00/3.55M [00:00<?, ?B/s]

Reducing the number of VLM layers to 16 ...


model.safetensors: reconstructing file:   0%|          |  0.00B /  907MB            

model.safetensors: downloading bytes:           |  0.00B            

Total parameters: 450046176
Trainable parameters: 99880992
[PASS] SmolVLA forward loss: 0.6225271821022034


In [4]:
import time

train_log = []
val_log = []
best_val_mae = float("inf")
best_val_fm = float("inf")

def a_noise(seed):
    g = torch.Generator(
        device=DEVICE
    ).manual_seed(int(seed))

    return torch.randn(
        (1, CHUNK, policy.config.max_action_dim),
        device=DEVICE,
        generator=g,
    )

@torch.no_grad()
def validate_a(step):
    policy.eval()

    maes = []
    losses = []

    for idx in VAL_INDICES:
        s = get_sample(ds, idx)
        b = a_batch(s)

        noise, tau = fixed_noise_time(
            900000 + idx,
            policy.config.max_action_dim
        )

        with torch.autocast("cuda", dtype=torch.float16):
            fm, _ = policy(
                b,
                noise=noise,
                time=tau,
            )

            pred = policy.predict_action_chunk(
                a_batch(s, include_action=False),
                noise=a_noise(700000 + idx),
            )[:, :, :DIM]

        pred_raw = pred.float() * AS + AM

        maes.append(
            (pred_raw - s["raw_action"]).abs()
            .mean().item()
        )
        losses.append(float(fm.item()))

    return {
        "val_mae": float(np.mean(maes)),
        "val_fm_loss": float(np.mean(losses)),
    }

def save_a_checkpoint(step, val_result):
    ckpt = {
        "model": "A",
        "base": SMOL_ID,
        "step": step,
        "val_mae": val_result["val_mae"],
        "val_fm_loss": val_result["val_fm_loss"],
        "trainable_state": {
            k: p.detach().cpu().clone()
            for k, p in policy.named_parameters()
            if p.requires_grad
        },
        "mean_action": AM.cpu(),
        "std_action": AS.cpu(),
        "mean_state": SM.cpu(),
        "std_state": SS.cpu(),
        "split_hash": SPLIT_HASH,
    }

    torch.save(ckpt, OUT / "A_best.pt")

torch.cuda.reset_peak_memory_stats()
train_start = time.perf_counter()
train_compute_seconds = 0.0

print("[TRAIN] SmolVLA A — 10000 steps", flush=True)

for step in range(1, STEPS + 1):
    torch.cuda.synchronize()
    t0 = time.perf_counter()

    item = TRAIN_PLAN[step - 1]
    s = get_sample(ds, item["index"])
    batch = a_batch(s)

    noise, tau = fixed_noise_time(
        step, policy.config.max_action_dim
    )

    policy.train()
    optim_a.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.float16):
        loss, _ = policy(
            batch,
            noise=noise,
            time=tau,
        )

    assert torch.isfinite(loss)

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        trainable,
        max_norm=1.0,
        error_if_nonfinite=True,
    )

    optim_a.step()

    torch.cuda.synchronize()
    step_seconds = time.perf_counter() - t0
    train_compute_seconds += step_seconds

    train_log.append({
        "model": "A",
        "step": step,
        "episode": item["episode"],
        "train_fm_loss": float(loss.item()),
        "step_seconds": step_seconds,
        "cumulative_train_seconds": train_compute_seconds,
        "steps_per_second": 1 / max(step_seconds, 1e-8),
    })

    if step == 1 or step % LOG_EVERY == 0:
        print(
            f"A {step}/{STEPS} | "
            f"loss={loss.item():.5f} | "
            f"s/step={step_seconds:.3f}",
            flush=True,
        )

        pd.DataFrame(train_log).to_csv(
            OUT / "A_training_steps.csv",
            index=False,
        )

    if step % VAL_EVERY == 0:
        scores = validate_a(step)
        best_val_fm = min(
            best_val_fm,
            scores["val_fm_loss"]
        )

        improved = scores["val_mae"] < best_val_mae

        if improved:
            best_val_mae = scores["val_mae"]
            save_a_checkpoint(step, scores)

        val_log.append({
            "model": "A",
            "step": step,
            **scores,
            "best_val_mae": best_val_mae,
            "best_val_fm_loss": best_val_fm,
            "selected": improved,
        })

        pd.DataFrame(val_log).to_csv(
            OUT / "A_validation_history.csv",
            index=False,
        )

        print(
            f"[A VAL] step={step} "
            f"MAE={scores['val_mae']:.5f} "
            f"FM={scores['val_fm_loss']:.5f} "
            f"bestMAE={best_val_mae:.5f}",
            flush=True,
        )

torch.cuda.synchronize()

a_training_wall = time.perf_counter() - train_start
a_peak_vram = torch.cuda.max_memory_allocated() / 2**30

a_summary = pd.DataFrame([{
    "model": "A",
    "steps": STEPS,
    "training_compute_seconds": train_compute_seconds,
    "training_wall_seconds": a_training_wall,
    "mean_seconds_per_step": train_compute_seconds / STEPS,
    "steps_per_second": STEPS / train_compute_seconds,
    "samples_per_second": STEPS / train_compute_seconds,
    "peak_train_vram_gib": a_peak_vram,
    "parameters_total": A_TOTAL,
    "parameters_trainable": A_TRAINABLE,
    "best_val_mae": best_val_mae,
    "best_val_fm_loss": best_val_fm,
}])

a_summary.to_csv(
    OUT / "A_training_summary.csv",
    index=False,
)

assert (OUT / "A_best.pt").exists()

print("[PASS] A TRAINING COMPLETE")
print(a_summary.to_string(index=False))

[TRAIN] SmolVLA A — 10000 steps
A 1/10000 | loss=0.34953 | s/step=0.650
A 100/10000 | loss=0.52343 | s/step=0.262
A 200/10000 | loss=0.21225 | s/step=0.264
A 300/10000 | loss=0.23363 | s/step=0.269
A 400/10000 | loss=0.83789 | s/step=0.269
A 500/10000 | loss=0.54365 | s/step=0.273
A 600/10000 | loss=0.37306 | s/step=0.270
A 700/10000 | loss=0.53556 | s/step=0.275
A 800/10000 | loss=0.91070 | s/step=0.271
A 900/10000 | loss=0.55799 | s/step=0.269
A 1000/10000 | loss=0.34540 | s/step=0.272
[A VAL] step=1000 MAE=13.20776 FM=0.46337 bestMAE=13.20776
A 1100/10000 | loss=0.18431 | s/step=0.270
A 1200/10000 | loss=0.32655 | s/step=0.265
A 1300/10000 | loss=0.15965 | s/step=0.289
A 1400/10000 | loss=0.63830 | s/step=0.279
A 1500/10000 | loss=0.21760 | s/step=0.272
A 1600/10000 | loss=0.39249 | s/step=0.269
A 1700/10000 | loss=0.54596 | s/step=0.269
A 1800/10000 | loss=0.18503 | s/step=0.277
A 1900/10000 | loss=0.23723 | s/step=0.269
A 2000/10000 | loss=0.10006 | s/step=0.268
[A VAL] step=2000 

In [5]:
import hashlib

# Restore BEST checkpoint.
saved = torch.load(
    OUT / "A_best.pt",
    map_location="cpu",
    weights_only=False,
)

assert saved["split_hash"] == SPLIT_HASH

report = policy.load_state_dict(
    saved["trainable_state"],
    strict=False,
)

assert not report.unexpected_keys
policy.eval()

ds_test = LeRobotDataset(
    DATASET_ID,
    episodes=TEST_EPS,
    delta_timestamps=deltas,
    revision="main",
    token=HF_TOKEN,
    download_videos=True,
)

test_table = ds_test.select_columns([
    "episode_index", "frame_index"
]).to_pandas().reset_index(drop=True)

test_plan = []

for ep in TEST_EPS:
    group = test_table[
        test_table.episode_index == ep
    ]

    last = int(group.frame_index.max())

    candidates = group[
        group.frame_index <= last - CHUNK + 1
    ]

    assert len(candidates) >= WINDOWS_PER_EP

    pos = np.linspace(
        0, len(candidates) - 1,
        WINDOWS_PER_EP + 2,
    )[1:-1].round().astype(int)

    for idx, row in candidates.iloc[pos].iterrows():
        test_plan.append({
            "episode": int(ep),
            "frame_index": int(row.frame_index),
            "index": int(idx),
        })

assert len(test_plan) == 300

pd.DataFrame(test_plan).to_csv(
    OUT / "test_plan.csv",
    index=False,
)

def action_hash(gt):
    array = (
        gt.detach().cpu().float()
        .contiguous().numpy()
    )
    return hashlib.sha256(array.tobytes()).hexdigest()

def action_metrics(pred, gt, action_std):
    pred = pred.float()
    gt = gt.float()

    err = pred - gt

    dp = pred[:, 1:] - pred[:, :-1]
    dg = gt[:, 1:] - gt[:, :-1]

    ddp = dp[:, 1:] - dp[:, :-1]
    ddg = dg[:, 1:] - dg[:, :-1]

    result = {
        "mae": err.abs().mean().item(),
        "mse": err.square().mean().item(),
        "normalized_mae": (
            err / action_std.reshape(1, 1, 6)
        ).abs().mean().item(),
        "delta_error": (
            dp - dg
        ).abs().mean().item(),
        "second_diff_error": (
            ddp - ddg
        ).abs().mean().item(),
        "pred_smoothness": dp.abs().mean().item(),
        "demo_smoothness": dg.abs().mean().item(),
    }

    for j in range(6):
        result[f"joint_{j}_mae"] = (
            err[..., j].abs().mean().item()
        )

    return result

def summarize_test(df, model):
    return {
        "model": model,
        "predictions": len(df),
        "test_mae": df.mae.mean(),
        "test_mse": df.mse.mean(),
        "test_rmse": np.sqrt(df.mse.mean()),
        "test_normalized_mae": df.normalized_mae.mean(),
        "test_delta_error": df.delta_error.mean(),
        "test_second_diff_error": df.second_diff_error.mean(),
        "test_smoothness_ratio": (
            df.pred_smoothness.mean()
            / max(df.demo_smoothness.mean(), 1e-8)
        ),
        "test_seed_std": df.seed_std.mean(),
        "test_fm_loss": df.fm_loss.mean(),
        "inference_latency_mean_ms": df.inference_ms.mean(),
        "inference_latency_p50_ms": df.inference_ms.median(),
        "inference_latency_p95_ms": df.inference_ms.quantile(0.95),
        "chunks_per_second": 1000 / df.inference_ms.mean(),
    }

def stable_seed(ep, frame, sid):
    return 50000 + ep * 100000 + frame * 10 + sid

# Reproducible warm-up, excluded from latency statistics.
warm = get_sample(ds_test, test_plan[0]["index"])

for _ in range(3):
    with torch.inference_mode(), torch.autocast(
        "cuda", dtype=torch.float16
    ):
        _ = policy.predict_action_chunk(
            a_batch(warm, include_action=False),
            noise=a_noise(42),
        )

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

test_records = []
trajectory_records = []

# Save the same three episode IDs for every model.
TRAJECTORY_EPS = TEST_EPS[:3]

for wi, item in enumerate(test_plan):
    s = get_sample(ds_test, item["index"])
    gt = s["raw_action"]
    b = a_batch(s)

    preds = []
    times = []
    fmlosses = []

    for sid in NOISE_SEEDS:
        seed = stable_seed(
            item["episode"],
            item["frame_index"],
            sid,
        )

        # FM test loss with fixed noise/time.
        noise, tau = fixed_noise_time(
            seed, policy.config.max_action_dim
        )

        with torch.inference_mode(), torch.autocast(
            "cuda", dtype=torch.float16
        ):
            fm, _ = policy(
                b,
                noise=noise,
                time=tau,
            )

        torch.cuda.synchronize()
        t0 = time.perf_counter()

        with torch.inference_mode(), torch.autocast(
            "cuda", dtype=torch.float16
        ):
            pred = policy.predict_action_chunk(
                a_batch(s, include_action=False),
                noise=a_noise(seed),
            )[:, :, :DIM]

        torch.cuda.synchronize()
        latency = (time.perf_counter() - t0) * 1000

        raw_pred = pred.float() * AS + AM

        preds.append(raw_pred.detach())
        times.append(latency)
        fmlosses.append(float(fm.item()))

    seed_std = torch.cat(
        preds, dim=0
    ).std(dim=0, unbiased=False).mean().item()

    for sid, pred in enumerate(preds):
        r = action_metrics(pred, gt, AS)

        r.update({
            "model": "A",
            "episode": item["episode"],
            "frame_index": item["frame_index"],
            "seed": sid,
            "gt_hash": action_hash(gt),
            "seed_std": seed_std,
            "inference_ms": times[sid],
            "fm_loss": fmlosses[sid],
        })

        test_records.append(r)

        # One representative window per episode, seed 0.
        if (
            item["episode"] in TRAJECTORY_EPS
            and sid == 0
            and wi == next(
                k for k, x in enumerate(test_plan)
                if x["episode"] == item["episode"]
            )
        ):
            p = pred[0].cpu().numpy()
            g = gt[0].cpu().numpy()

            for t in range(CHUNK):
                for joint in range(DIM):
                    trajectory_records.append({
                        "model": "A",
                        "episode": item["episode"],
                        "frame_index": item["frame_index"],
                        "timestep": t,
                        "joint": joint,
                        "prediction": float(p[t, joint]),
                        "ground_truth": float(g[t, joint]),
                    })

    if (wi + 1) % 30 == 0:
        print(
            f"[A TEST] {wi+1}/300 windows",
            flush=True,
        )

        pd.DataFrame(test_records).to_csv(
            OUT / "A_test_details.csv",
            index=False,
        )

df = pd.DataFrame(test_records)
assert len(df) == 900

test_summary = pd.DataFrame([
    summarize_test(df, "A")
])

test_summary["peak_inference_vram_gib"] = (
    torch.cuda.max_memory_allocated() / 2**30
)

test_summary.to_csv(
    OUT / "A_test_summary.csv",
    index=False,
)

df.to_csv(OUT / "A_test_details.csv", index=False)

df.groupby("episode", as_index=False).agg(
    mae=("mae", "mean"),
    mse=("mse", "mean"),
    fm_loss=("fm_loss", "mean"),
).to_csv(
    OUT / "A_test_per_episode.csv",
    index=False,
)

pd.DataFrame(trajectory_records).to_csv(
    OUT / "A_action_trajectories.csv",
    index=False,
)

print("\n[PASS] SMOLVLA TEST COMPLETE")
print(test_summary.to_string(index=False))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[A TEST] 30/300 windows
[A TEST] 60/300 windows
[A TEST] 90/300 windows
[A TEST] 120/300 windows
[A TEST] 150/300 windows
[A TEST] 180/300 windows
[A TEST] 210/300 windows
[A TEST] 240/300 windows
[A TEST] 270/300 windows
[A TEST] 300/300 windows

[PASS] SMOLVLA TEST COMPLETE
model  predictions  test_mae   test_mse  test_rmse  test_normalized_mae  test_delta_error  test_second_diff_error  test_smoothness_ratio  test_seed_std  test_fm_loss  inference_latency_mean_ms  inference_latency_p50_ms  inference_latency_p95_ms  chunks_per_second  peak_inference_vram_gib
    A          900  9.131345 202.961653  14.246461             0.305293          2.177532                3.480496               5.138217       3.837623      0.240545                 394.097762                392.938403                406.382057           2.537441                 2.290751
